In [1]:
!pip install crewai==0.28.8 \
            crewai_tools==0.1.6 \
            langchain==0.1.14 \
            langchain_community==0.0.30 \
            openai==1.25.1 \
            httpx==0.27.0 \
            python-dotenv

In [2]:
import pandas as pd
import numpy as np

PATHS = {
    "production_plan": "./A_company_crewai_demo_csvs/production_plan.csv",
    "process_master": "./A_company_crewai_demo_csvs/process_master.csv",
    "line_assignment": "./A_company_crewai_demo_csvs/line_assignment.csv",
    "work_event_log": "./A_company_crewai_demo_csvs/work_event_log.csv",
    "historical_ct": "./A_company_crewai_demo_csvs/historical_ct.csv",
    "delay_history": "./A_company_crewai_demo_csvs/delay_history.csv",
    "real_time_production": "./A_company_crewai_demo_csvs/real_time_production.csv",
}

dfs = {k: pd.read_csv(v) for k, v in PATHS.items()}

plan = dfs["production_plan"].copy()
pm   = dfs["process_master"].copy()
la   = dfs["line_assignment"].copy()
wel  = dfs["work_event_log"].copy()
hct  = dfs["historical_ct"].copy()
dh   = dfs["delay_history"].copy()
rt   = dfs["real_time_production"].copy()

# 타입 정리
plan["due_time"] = pd.to_datetime(plan["due_time"])
wel["event_time"] = pd.to_datetime(wel["event_time"])
la["planned_start"] = pd.to_datetime(la["planned_start"])
la["planned_end"]   = pd.to_datetime(la["planned_end"])
rt["time"] = pd.to_datetime(rt["time"])

plan


,date,shift,po_id,sku,sku_name,target_qty,due_time,priority,customer_grade,notes
0,2026-01-15,DAY,PO20260115-004,SKU-SRM-30,세럼 30ml,300,2026-01-15 14:00:00,1,B,NaN
1,2026-01-15,DAY,PO20260115-002,SKU-TNR-200,토너 200ml,360,2026-01-15 15:00:00,1,A,NaN
2,2026-01-15,DAY,PO20260115-001,SKU-CLN-100,클렌저 100ml,300,2026-01-15 14:00:00,3,A,NaN
3,2026-01-15,DAY,PO20260115-003,SKU-CRM-50,크림 50ml,540,2026-01-15 14:00:00,3,C,NaN
4,2026-01-15,DAY,PO20260115-005,SKU-SPF-50,선크림 50ml,360,2026-01-15 14:00:00,3,B,NaN


In [3]:
from dataclasses import dataclass
from typing import Dict, Any, List, Optional

STEP_ORDER = (
    pm[["sku","step_id","step_order","step_name","std_ct_sec"]]
    .drop_duplicates()
)

def get_snapshot(snapshot_time: Optional[pd.Timestamp] = None) -> pd.DataFrame:
    """15분 단위 스냅샷에서 특정 시점(없으면 최신) 추출"""
    if snapshot_time is None:
        snapshot_time = rt["time"].max()
    snap = rt[rt["time"] == snapshot_time].copy()
    return snap, snapshot_time

def get_last_step_state(po_id: str, asof: pd.Timestamp) -> Dict[str, Any]:
    """event log 기반: 특정 PO가 asof 시점에 어떤 step에서 어떤 상태인지 추정"""
    sub = wel[(wel["po_id"] == po_id) & (wel["event_time"] <= asof)].sort_values("event_time")
    if sub.empty:
        return {"po_id": po_id, "state": "NO_LOG"}

    last_by_step = sub.groupby("step_id").tail(1)

    # 진행 중(step ongoing) 판단: 마지막 이벤트가 start/resume/pause 이고 이후 end가 없으면 ongoing
    ongoing = []
    for _, r in last_by_step.iterrows():
        if r["event_type"] in ("start","resume","pause"):
            ended = sub[(sub["step_id"] == r["step_id"]) & (sub["event_type"] == "end")]
            if ended.empty or ended["event_time"].max() < r["event_time"]:
                ongoing.append(r)

    sku = sub.iloc[-1]["sku"]
    om = STEP_ORDER[STEP_ORDER["sku"] == sku].set_index("step_id")["step_order"].to_dict()

    if ongoing:
        ongoing.sort(key=lambda r: om.get(r["step_id"], 0), reverse=True)
        r = ongoing[0]
        return {
            "po_id": po_id,
            "sku": r["sku"],
            "line_id": r["line_id"],
            "step_id": r["step_id"],
            "event_type": r["event_type"],
            "event_time": r["event_time"],
            "operator_id": r["operator_id"],
            "memo": r.get("memo", None),
        }

    # ongoing 없으면 마지막 end 기준으로 "최근 완료 step"
    r = sub[sub["event_type"] == "end"].tail(1)
    if r.empty:
        r = sub.tail(1)
    r = r.iloc[0]
    return {
        "po_id": po_id,
        "sku": r["sku"],
        "line_id": r["line_id"],
        "step_id": r["step_id"],
        "event_type": "end",
        "event_time": r["event_time"],
        "operator_id": r["operator_id"],
        "memo": r.get("memo", None),
    }

def detect_delayed_start(asof: pd.Timestamp, threshold_min: int = 10) -> pd.DataFrame:
    """
    계획(planned_start) 대비 실제 start가 늦은 step 감지
    """
    # 실제 start 시간
    starts = wel[wel["event_type"] == "start"][["po_id","step_id","event_time"]].rename(columns={"event_time":"actual_start"})
    merged = la.merge(starts, on=["po_id","step_id"], how="left")

    merged = merged[merged["planned_start"] <= asof].copy()
    merged["delay_min"] = (merged["actual_start"] - merged["planned_start"]).dt.total_seconds() / 60
    merged["delay_min"] = merged["delay_min"].fillna(np.nan)

    out = merged[(merged["actual_start"].isna()) | (merged["delay_min"] >= threshold_min)].copy()
    out = out.sort_values(["po_id","planned_start"])
    return out[["po_id","sku","line_id","step_id","operator_id","planned_start","actual_start","delay_min"]]

def detect_pause(asof: pd.Timestamp) -> pd.DataFrame:
    """pause 상태 PO/step 감지"""
    rows = []
    for po in plan["po_id"].unique():
        st = get_last_step_state(po, asof)
        if st.get("event_type") == "pause":
            rows.append(st)
    return pd.DataFrame(rows)

def build_risk_ranking(asof: pd.Timestamp, top_n: int = 5) -> pd.DataFrame:
    """
    간단 PoC용 리스크 스코어:
    - progress 낮을수록 +
    - due_time 임박/초과일수록 +
    - pause면 크게 +
    - priority(1이 제일 중요) 반영
    """
    snap, _ = get_snapshot(asof)
    snap_map = snap.set_index("po_id")[["progress_pct","cumulative_qty","target_qty","status"]]

    rows = []
    for _, r in plan.iterrows():
        po = r["po_id"]
        due = r["due_time"]
        min_to_due = (due - asof).total_seconds()/60

        if po in snap_map.index:
            prog = float(snap_map.loc[po, "progress_pct"])
        else:
            prog = np.nan

        st = get_last_step_state(po, asof)
        paused = (st.get("event_type") == "pause")

        base = (100 - prog) if not np.isnan(prog) else 30
        due_pen = max(0, -min_to_due) + max(0, 30 - min_to_due)  # 이미 늦음 + 30분 이내 임박
        pause_pen = 50 if paused else 0
        pri_pen = (4 - int(r["priority"])) * 5  # priority=1이면 15, priority=3이면 5

        score = base + due_pen + pause_pen + pri_pen

        rows.append({
            "po_id": po,
            "sku": r["sku"],
            "due_time": due,
            "min_to_due": round(min_to_due, 1),
            "progress_pct": None if np.isnan(prog) else round(prog, 1),
            "current_step": st.get("step_id"),
            "current_event": st.get("event_type"),
            "line_id": st.get("line_id"),
            "operator_id": st.get("operator_id"),
            "memo": st.get("memo"),
            "risk_score": round(score, 1),
        })

    out = pd.DataFrame(rows).sort_values("risk_score", ascending=False).head(top_n)
    return out

def summarize_field_status(asof: pd.Timestamp) -> str:
    """관리자가 보는 현장요약 텍스트(웹 UI 카드에 바로 띄울 정도)"""
    snap, asof = get_snapshot(asof)
    lines = []
    lines.append(f"[현장 요약] 기준시각: {asof:%Y-%m-%d %H:%M}")
    for _, r in snap.sort_values("progress_pct").iterrows():
        po = r["po_id"]
        st = get_last_step_state(po, asof)
        step_name = STEP_ORDER[(STEP_ORDER["sku"]==r["sku"]) & (STEP_ORDER["step_id"]==st.get("step_id"))]["step_name"]
        step_name = step_name.iloc[0] if len(step_name) else st.get("step_id")
        memo = st.get("memo")
        memo_txt = f" / 메모: {memo}" if isinstance(memo, str) and memo.strip() else ""
        lines.append(
            f"- {po}({r['sku_name']}): {r['progress_pct']:.1f}% / 현재: {step_name}({st.get('event_type')})"
            f" / 라인:{st.get('line_id')} / 작업자:{st.get('operator_id')}{memo_txt}"
        )
    return "\n".join(lines)


In [4]:
ASOF = pd.Timestamp("2026-01-15 14:00:00")

print(summarize_field_status(ASOF))

print("\n[지연 시작 감지 - planned_start 대비]")
display(detect_delayed_start(ASOF, threshold_min=10))

print("\n[PAUSE 감지]")
display(detect_pause(ASOF))

print("\n[리스크 TOP 랭킹]")
display(build_risk_ranking(ASOF, top_n=5))


[현장 요약] 기준시각: 2026-01-15 14:00
- PO20260115-003(크림 50ml): 77.8% / 현재: 충진(Filling)(pause) / 라인:L2-충진 / 작업자:OP08 / 메모: 원료 용기 부족(자재 지연)
- PO20260115-001(클렌저 100ml): 100.0% / 현재: 라벨링(start) / 라인:L3-포장 / 작업자:OP04
- PO20260115-002(토너 200ml): 100.0% / 현재: 라벨링(start) / 라인:L3-포장 / 작업자:OP04
- PO20260115-004(세럼 30ml): 100.0% / 현재: 캡핑/실링(start) / 라인:L2-충진 / 작업자:OP03
- PO20260115-005(선크림 50ml): 100.0% / 현재: 캡핑/실링(start) / 라인:L2-충진 / 작업자:OP08

[지연 시작 감지 - planned_start 대비]


,po_id,sku,line_id,step_id,operator_id,planned_start,actual_start,delay_min



[PAUSE 감지]


,po_id,sku,line_id,step_id,event_type,event_time,operator_id,memo
0,PO20260115-003,SKU-CRM-50,L2-충진,S30,pause,2026-01-15 13:53:00,OP08,원료 용기 부족(자재 지연)



[리스크 TOP 랭킹]


,po_id,sku,due_time,min_to_due,progress_pct,current_step,current_event,line_id,operator_id,memo,risk_score
3,PO20260115-003,SKU-CRM-50,2026-01-15 14:00:00,0.0,77.8,S30,pause,L2-충진,OP08,원료 용기 부족(자재 지연),107.2
0,PO20260115-004,SKU-SRM-30,2026-01-15 14:00:00,0.0,100.0,S40,start,L2-충진,OP03,NaN,45.0
2,PO20260115-001,SKU-CLN-100,2026-01-15 14:00:00,0.0,100.0,S50,start,L3-포장,OP04,NaN,35.0
4,PO20260115-005,SKU-SPF-50,2026-01-15 14:00:00,0.0,100.0,S40,start,L2-충진,OP08,NaN,35.0
1,PO20260115-002,SKU-TNR-200,2026-01-15 15:00:00,60.0,100.0,S50,start,L3-포장,OP04,NaN,15.0


In [6]:
import os
import openai

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

api_key = os.getenv("OPENAI_API_KEY")  # .env 파일에서 로드 # 개인 API 키를 입력합니다.

# 키가 없을 경우 예외 발생
if not api_key:
    raise ValueError("API 키가 비어 있습니다. 올바른 키를 입력하세요.")

os.environ["OPENAI_API_KEY"] = api_key
os.environ["OPENAI_MODEL_NAME"] = 'gpt-4o'

In [7]:
from crewai_tools import tool
import pandas as pd

# 전역 asof 주입
ASOF_FOR_TOOLS = None

def df_to_markdown(df: pd.DataFrame, max_rows: int = 12) -> str:
    """DataFrame -> Markdown table (너무 길면 head로 컷)"""
    if df is None or len(df) == 0:
        return "_(데이터 없음)_"
    show = df.head(max_rows).copy()
    return show.to_markdown(index=False)

@tool("report_inputs")
def report_inputs_tool() -> str:
    """
    에이전트가 참고할 수 있도록 핵심 근거를 텍스트로 제공:
    - 현장 상태 요약
    - 지연 시작/미시작
    - pause 상태
    - 리스크 랭킹
    """
    asof = ASOF_FOR_TOOLS
    s1 = summarize_field_status(asof)

    d1 = detect_delayed_start(asof, threshold_min=10)
    p1 = detect_pause(asof)
    r1 = build_risk_ranking(asof, top_n=10)

    out = []
    out.append("## [근거] 현장요약")
    out.append(s1)
    out.append("\n## [근거] 지연 시작/미시작(계획 대비)")
    out.append(df_to_markdown(d1, max_rows=10))
    out.append("\n## [근거] PAUSE 감지")
    out.append(df_to_markdown(p1, max_rows=10))
    out.append("\n## [근거] 리스크 TOP10")
    out.append(df_to_markdown(r1, max_rows=10))
    return "\n".join(out)

@tool("tables_pack")
def tables_pack_tool() -> str:
    """
    보고서용 '표 패키지'를 Markdown 표로 반환.
    (1) 리스크 TOP
    (2) 지연/미시작 공정
    (3) pause 공정
    (4) PO별 현상태(요약 표)
    """
    asof = ASOF_FOR_TOOLS

    # (1) 리스크 TOP
    risk = build_risk_ranking(asof, top_n=8)

    # (2) 지연/미시작
    delayed = detect_delayed_start(asof, threshold_min=10)

    # (3) pause
    paused = detect_pause(asof)

    # (4) PO별 현상태 요약표
    # plan의 po 리스트 기준으로 last_step 상태를 표로 만든다
    rows = []
    for po in plan["po_id"].unique():
        st = get_last_step_state(po, asof)
        rows.append({
            "po_id": po,
            "sku": st.get("sku"),
            "line_id": st.get("line_id"),
            "step_id": st.get("step_id"),
            "event": st.get("event_type"),
            "operator_id": st.get("operator_id"),
            "event_time": st.get("event_time"),
            "memo": st.get("memo"),
        })
    status_df = pd.DataFrame(rows).sort_values(["line_id","po_id"])

    out = []
    out.append(f"### 기준시각: {asof:%Y-%m-%d %H:%M}\n")

    out.append("## 1) 리스크 TOP")
    out.append(df_to_markdown(risk, max_rows=8))

    out.append("\n## 2) 지연 시작/미시작 공정")
    out.append(df_to_markdown(delayed, max_rows=10))

    out.append("\n## 3) PAUSE 공정")
    out.append(df_to_markdown(paused, max_rows=10))

    out.append("\n## 4) PO별 현장 상태 요약")
    out.append(df_to_markdown(status_df, max_rows=12))

    return "\n".join(out)

@tool("alert_lines")
def alert_lines_tool() -> str:
    """
    알림 로그를 [알림] 형태로 생성하기 위한 '팩트'를 제공.
    LLM이 이를 바탕으로 최종 알림 문구를 작성한다.
    """
    asof = ASOF_FOR_TOOLS
    delayed = detect_delayed_start(asof, threshold_min=10)
    paused = detect_pause(asof)
    risk = build_risk_ranking(asof, top_n=5)

    # 간단 팩트 요약(LLM이 문구 만들기 쉽게)
    facts = []
    facts.append(f"기준시각={asof:%Y-%m-%d %H:%M}")

    facts.append(f"지연/미시작 건수={len(delayed)}")
    if len(delayed):
        top = delayed.iloc[0].to_dict()
        facts.append(f"지연대표={top}")

    facts.append(f"pause 건수={len(paused)}")
    if len(paused):
        top = paused.iloc[0].to_dict()
        facts.append(f"pause대표={top}")

    if len(risk):
        facts.append("리스크TOP5=" + risk[["po_id","risk_score","min_to_due","current_step","current_event","line_id"]].to_string(index=False))

    return "\n".join([str(x) for x in facts])


In [10]:
import re
from IPython.display import display, Markdown

def strip_code_fences(text: str) -> str:
    """
    LLM이 ```markdown ...``` 같은 코드펜스를 씌워버릴 때 제거.
    """
    if not isinstance(text, str):
        text = str(text)

    # ```markdown ... ``` 혹은 ``` ... ``` 제거
    text = re.sub(r"^```(?:markdown)?\s*", "", text.strip(), flags=re.IGNORECASE)
    text = re.sub(r"\s*```$", "", text.strip())
    return text.strip()


In [15]:
from crewai import Agent, Task, Crew, Process

def run_supervisor_report_crew(asof):
    global ASOF_FOR_TOOLS
    ASOF_FOR_TOOLS = asof

    # 🚨 출력 규칙(코드펜스 금지 + 표는 마크다운 테이블만)
    OUTPUT_RULES = """
- 절대 ``` 또는 ```markdown 같은 코드펜스(백틱 3개)를 사용하지 마라.
- 결과는 '순수 마크다운'만 출력하라. (그대로 렌더링되어야 한다.)
- 표는 반드시 마크다운 테이블 형식(|---|---|)으로 출력하라.
- 불필요한 설명/서론 최소화. 현장 보고서처럼 간결하게.
"""

    report_table_agent = Agent(
        role="운영 리포트 표 생성 Agent",
        goal="생산관리자가 한눈에 보도록, 상황에 맞는 Markdown 표(리스크/지연/PAUSE/현장요약)를 생성한다.",
        backstory="현장 데이터를 표로 정리하여 관리자의 판단 시간을 줄이는 리포팅 전문가.",
        allow_delegation=False,
        verbose=True,
        tools=[tables_pack_tool],
    )

    notifier_agent = Agent(
        role="슈퍼바이저 알림 전송 Agent",
        goal="지연/누락/정체 징후를 읽고, 즉시 조치가 필요한 대상과 내용을 [알림] 로그 형태로 작성한다.",
        backstory="라인단장과 생산관리자에게 '지금 당장 확인할 것'만 짧고 강하게 전달하는 현장 메신저.",
        allow_delegation=False,
        verbose=True,
        tools=[alert_lines_tool],
    )

    editor_agent = Agent(
        role="최종 보고서 편집 Agent",
        goal="표와 알림을 합쳐 최종 Markdown 보고서 형태로 정리하고, 조치 우선순위를 명확히 제시한다.",
        backstory="실무 보고서처럼 읽히는 포맷으로 정리하는 편집자.",
        allow_delegation=False,
        verbose=True,
        tools=[report_inputs_tool],
    )

    # ---- Tasks ----
    t_table = Task(
        description=(
            OUTPUT_RULES +
            "\n\n[업무]\n"
            "tables_pack tool을 사용해 보고서용 Markdown 표를 생성하라.\n"
            "반드시 아래 섹션 순서를 지켜라:\n"
            "## 1) 리스크 TOP\n"
            "## 2) 지연 시작/미시작 공정\n"
            "## 3) PAUSE 공정\n"
            "## 4) PO별 현장 상태 요약\n"
            "각 섹션 아래에는 반드시 마크다운 테이블이 있어야 한다.\n"
        ),
        expected_output="순수 마크다운 섹션 + 마크다운 테이블 4개 (코드펜스 금지)",
        agent=report_table_agent,
    )

    t_alert = Task(
        description=(
            OUTPUT_RULES +
            "\n\n[업무]\n"
            "alert_lines tool로 팩트를 얻고, 아래 형식으로 '알림 로그'를 생성하라.\n"
            "형식(예시):\n"
            "- [알림] 0라인 : ~~ 확인 필요 -> ~~ 조치 권장. 수신: 라인단장(Lxx), 생산관리자\n"
            "- [알림] 1라인 : ~~\n"
            "- [알림] 2라인 : ~~\n"
            "규칙:\n"
            "1) 3~6줄\n"
            "2) 지연/미시작이 있으면 반드시 포함\n"
            "3) PAUSE가 있으면 반드시 포함\n"
            "4) due 임박/초과(min_to_due 작거나 음수)면 반드시 포함\n"
        ),
        expected_output="[알림] 로그 3~6줄 (코드펜스 금지)",
        agent=notifier_agent,
    )

    t_final = Task(
        description=(
            OUTPUT_RULES +
            "\n\n[업무]\n"
            "report_inputs tool로 근거를 확인한 뒤, 아래 최종 보고서를 '순수 마크다운'으로 작성하라.\n\n"
            "필수 구성:\n"
            "## A. 알림 로그\n"
            "(t_alert 결과를 그대로 붙여라)\n\n"
            "## B. 운영 리포트 표\n"
            "(t_table 결과를 그대로 붙여라)\n\n"
            "## C. 즉시 조치 TOP3\n"
            "- '대상(PO/라인/공정) / 이유 / 다음 액션(1문장)' 형태로 3개\n\n"
            "주의:\n"
            "- 절대 코드펜스로 감싸지 마라.\n"
            "- 마크다운 테이블이 깨지지 않게 그대로 유지하라.\n"
        ),
        expected_output="최종 순수 마크다운 보고서 (코드펜스 금지, 표 유지)",
        agent=editor_agent,
        context=[t_alert, t_table],
    )

    crew = Crew(
        agents=[report_table_agent, notifier_agent, editor_agent],
        tasks=[t_table, t_alert, t_final],
        process=Process.sequential,
        verbose=True,
    )

    result = crew.kickoff()
    # 혹시라도 모델이 ```를 붙이면 제거
    result_text = strip_code_fences(str(result))
    return result_text

final_report_md = run_supervisor_report_crew(ASOF)
display(Markdown(final_report_md))


2026-01-15 12:16:36,582 - 129303251140736 - __init__.py-__init__:537 - WARNING: Overriding of current TracerProvider is not allowed


 [DEBUG]: == Working Agent: 운영 리포트 표 생성 Agent
 [INFO]: == Starting Task: 
- 절대 ``` 또는 ```markdown 같은 코드펜스(백틱 3개)를 사용하지 마라.
- 결과는 '순수 마크다운'만 출력하라. (그대로 렌더링되어야 한다.)
- 표는 반드시 마크다운 테이블 형식(|---|---|)으로 출력하라.
- 불필요한 설명/서론 최소화. 현장 보고서처럼 간결하게.


[업무]
tables_pack tool을 사용해 보고서용 Markdown 표를 생성하라.
반드시 아래 섹션 순서를 지켜라:
## 1) 리스크 TOP
## 2) 지연 시작/미시작 공정
## 3) PAUSE 공정
## 4) PO별 현장 상태 요약
각 섹션 아래에는 반드시 마크다운 테이블이 있어야 한다.



> Entering new CrewAgentExecutor chain...
Action: tables_pack
Action Input: {} 

### 기준시각: 2026-01-15 14:00

## 1) 리스크 TOP
| po_id          | sku         | due_time            |   min_to_due |   progress_pct | current_step   | current_event   | line_id   | operator_id   | memo                      |   risk_score |
|:---------------|:------------|:--------------------|-------------:|---------------:|:---------------|:----------------|:----------|:--------------|:--------------------------|-------------:|
| PO20260115-003 | SKU-CRM-50  | 2026-01-15 14:00:00 |            0 |           77

Agent stopped due to iteration limit or time limit.